# Funda Real Estate Analysis

This notebook explores Amsterdam listings on Funda using `pyfunda`, and also inspects the raw search totals that the website search uses.

It answers two different questions:

- How many listings does Funda currently show for `amsterdam`?
- What do price, size, neighbourhood, and object-type distributions look like in a larger sample?

## Setup

Run the next cell once inside the notebook kernel.


In [ ]:
%pip install pyfunda pandas matplotlib

In [ ]:
import json
import math
import random
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from funda import Funda
import funda.funda as ff

plt.rcParams['figure.figsize'] = (10, 6)
pd.options.display.max_columns = 20
pd.options.display.float_format = '{:,.0f}'.format

MISSING = object()
RESULTS_PER_PAGE = 15
REPO_DEFAULT_OBJECT_TYPES = ['house', 'apartment']


def _search_payload(location, page=0, object_type=MISSING, availability=None, offering_type='buy'):
    params = {
        'availability': availability or ['available', 'negotiations'],
        'type': ['single'],
        'zoning': ['residential'],
        'publication_date': {'no_preference': True},
        'offering_type': offering_type,
        'page': {'from': page * RESULTS_PER_PAGE},
        'sort': {'field': None, 'order': None},
        'selected_area': [location],
    }
    if object_type is not MISSING:
        params['object_type'] = object_type

    index_line = json.dumps({'index': 'listings-wonen-searcher-alias-prod'})
    query_line = json.dumps({'id': 'search_result_20250805', 'params': params})
    return f'{index_line}\n{query_line}\n'


def search_page(f, location, page=0, object_type=MISSING, availability=None, offering_type='buy', context='search'):
    payload = _search_payload(
        location,
        page=page,
        object_type=object_type,
        availability=availability,
        offering_type=offering_type,
    )
    for attempt in range(3):
        response = f._post(ff.API_SEARCH, ff._make_headers(for_search=True), data=payload, for_search=True)
        if response.status_code == 200:
            return response.json()
        if response.status_code == 400 and attempt < 2:
            time.sleep(0.25 * (attempt + 1))
            continue
        raise RuntimeError(f'{context} failed (status {response.status_code})')


def search_total(location, object_type=MISSING, availability=None, offering_type='buy', retry_forever=True):
    cooldown = 30
    attempt = 0
    while True:
        attempt += 1
        try:
            with Funda() as f:
                data = search_page(
                    f,
                    location,
                    page=0,
                    object_type=object_type,
                    availability=availability,
                    offering_type=offering_type,
                    context=f'total lookup for {location}',
                )
                return data['responses'][0]['hits']['total']['value']
        except RuntimeError as exc:
            if '403' not in str(exc) or not retry_forever:
                raise
            wait_seconds = min(cooldown * (2 ** min(attempt - 1, 5)), 900)
            print(f'Total lookup blocked for {location}. Waiting {wait_seconds}s before retrying...')
            time.sleep(wait_seconds)


def fetch_page(location, page, object_type=MISSING, availability=None, offering_type='buy'):
    with Funda() as f:
        data = search_page(
            f,
            location,
            page=page,
            object_type=object_type,
            availability=availability,
            offering_type=offering_type,
            context=f'page {page + 1} for {location}',
        )
        return [listing.to_dict() for listing in f._parse_search_results(data)]


def load_checkpoint(checkpoint_path):
    path = Path(checkpoint_path)
    if not path.exists():
        return []

    rows = []
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return rows


def fetch_listings(
    location,
    pages,
    object_type=MISSING,
    availability=None,
    offering_type='buy',
    checkpoint_path=None,
    min_delay=4.0,
    max_delay=8.0,
    cooldown_base=60,
    max_403_retries=None,
    long_pause_every=10,
    long_pause_seconds=90,
):
    rows = load_checkpoint(checkpoint_path) if checkpoint_path else []
    completed_pages = {row.get('_page') for row in rows if '_page' in row}
    consecutive_403 = 0

    if checkpoint_path:
        print(f'Loaded {len(rows):,} rows from {checkpoint_path}')

    for page in range(pages):
        if page in completed_pages:
            continue

        while True:
            try:
                page_rows = fetch_page(
                    location,
                    page,
                    object_type=object_type,
                    availability=availability,
                    offering_type=offering_type,
                )
                consecutive_403 = 0
                break
            except RuntimeError as exc:
                if '403' not in str(exc):
                    raise

                consecutive_403 += 1
                if max_403_retries is not None and consecutive_403 > max_403_retries:
                    raise RuntimeError(
                        f'Funda kept returning 403 after page {page}. '
                        f'Progress is saved in {checkpoint_path}. Wait a while and rerun the cell to resume.'
                    ) from exc

                wait_seconds = min(cooldown_base * (2 ** (consecutive_403 - 1)), 900)
                print(f'403 on page {page + 1}/{pages}. Cooling down for {wait_seconds}s before retrying...')
                time.sleep(wait_seconds)

        if not page_rows:
            break

        annotated_rows = []
        for row in page_rows:
            row = dict(row)
            row['_page'] = page
            annotated_rows.append(row)

        rows.extend(annotated_rows)
        completed_pages.add(page)

        if checkpoint_path:
            with Path(checkpoint_path).open('a', encoding='utf-8') as handle:
                for row in annotated_rows:
                    handle.write(json.dumps(row, ensure_ascii=True) + '\\n')

        print(f'Fetched page {page + 1}/{pages}: +{len(page_rows)} listings (total {len(rows):,})')
        if long_pause_every and (page + 1) % long_pause_every == 0 and page + 1 < pages:
            print(f'Cooling down after {page + 1} pages for {long_pause_seconds}s...')
            time.sleep(long_pause_seconds)
        else:
            time.sleep(random.uniform(min_delay, max_delay))

    return rows


## Amsterdam Search Coverage

`pyfunda.search_listing('amsterdam')` applies an `object_type=['house', 'apartment']` filter by default.

The cells below compare that default with the broader website-style residential search, which omits that object type filter.

If you set `FETCH_ALL_PAGES = True`, the notebook will crawl every Amsterdam page, write progress to a checkpoint file, and resume cleanly after a 403 cooldown.


In [ ]:
LOCATION = 'amsterdam'
FETCH_ALL_PAGES = True
MAX_SAMPLE_PAGES = 10
CHECKPOINT_PATH = 'amsterdam_listings_checkpoint.jsonl'

repo_total = search_total(LOCATION, object_type=REPO_DEFAULT_OBJECT_TYPES)
website_total = search_total(LOCATION)

target_pages = math.ceil(website_total / RESULTS_PER_PAGE) if FETCH_ALL_PAGES else min(MAX_SAMPLE_PAGES, math.ceil(website_total / RESULTS_PER_PAGE))

all_listings = fetch_listings(
    LOCATION,
    pages=target_pages,
    checkpoint_path=CHECKPOINT_PATH,
)

coverage = pd.DataFrame([
    {'search_variant': 'pyfunda default (house + apartment)', 'total_listings': repo_total},
    {'search_variant': 'website-style residential search', 'total_listings': website_total},
    {'search_variant': f'notebook fetched ({target_pages} pages requested)', 'total_listings': len(all_listings)},
])
coverage['share_of_website_total_pct'] = (coverage['total_listings'] / website_total * 100).round(1)

coverage


In [ ]:
print(f"Website-style total for {LOCATION}: {website_total:,} listings")
print(f"pyfunda default total for {LOCATION}: {repo_total:,} listings")
print(f"Notebook fetched: {len(all_listings):,} listings across {target_pages} requested pages")
print(f"Difference caused by pyfunda's default object_type filter: {website_total - repo_total:,} listings")
print(f"Checkpoint file: {CHECKPOINT_PATH}")

## Amsterdam Sample

The analysis below uses the broader website-style residential search and works on the sampled listings fetched above.


In [ ]:
df = pd.DataFrame(all_listings)
df = df.drop_duplicates(subset=['global_id']) if 'global_id' in df.columns else df
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['living_area'] = pd.to_numeric(df['living_area'], errors='coerce')
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
df['price_per_m2'] = df['price'] / df['living_area']

cols = [
    'title', 'city', 'neighbourhood', 'postcode', 'price', 'living_area', 'price_per_m2',
    'bedrooms', 'energy_label', 'object_type', 'status', 'url'
]
df = df[[c for c in cols if c in df.columns]].sort_values('price', ascending=False)
df.head(10)


In [ ]:
summary = pd.Series({
    'sample_listings': len(df),
    'median_price_eur': df['price'].median(),
    'mean_price_eur': df['price'].mean(),
    'p90_price_eur': df['price'].quantile(0.90),
    'median_living_area_m2': df['living_area'].median(),
    'median_price_per_m2': df['price_per_m2'].median(),
    'median_bedrooms': df['bedrooms'].median(),
}).to_frame('value')
summary


## Price Distribution

In [ ]:
fig, ax = plt.subplots()
df['price'].dropna().hist(bins=25, ax=ax, edgecolor='black', color='steelblue')
ax.set_xlabel('Price (EUR)')
ax.set_ylabel('Listings in sample')
ax.set_title(f'Price Distribution in {LOCATION.title()}')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()


In [ ]:
expensive = df.nlargest(10, 'price')[['title', 'neighbourhood', 'price', 'living_area', 'price_per_m2', 'object_type']]
expensive


## Price per m2

In [ ]:
df_area = df[df['living_area'].notna() & (df['living_area'] > 0) & df['price'].notna()].copy()

print(f"Mean price/m2:   {df_area['price_per_m2'].mean():,.0f}")
print(f"Median price/m2: {df_area['price_per_m2'].median():,.0f}")
print(f"P90 price/m2:    {df_area['price_per_m2'].quantile(0.90):,.0f}")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df_area['living_area'], df_area['price'], alpha=0.6, color='darkorange')
ax.set_xlabel('Living Area (m2)')
ax.set_ylabel('Price (EUR)')
ax.set_title('Price vs Living Area')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()


## Listing Mix

In [ ]:
object_mix = df['object_type'].value_counts(dropna=False).rename_axis('object_type').to_frame('listings')
energy_mix = df['energy_label'].fillna('Unknown').value_counts().rename_axis('energy_label').to_frame('listings')

display(object_mix)
display(energy_mix.head(10))


In [ ]:
neighbourhoods = (
    df.dropna(subset=['neighbourhood', 'price'])
      .groupby('neighbourhood')
      .agg(listings=('price', 'size'), median_price=('price', 'median'), median_price_per_m2=('price_per_m2', 'median'))
      .query('listings >= 3')
      .sort_values('median_price', ascending=False)
)

neighbourhoods.head(10)


## Compare Cities

Use the broader website-style residential search for each city, then compare both true search totals and median asking prices from a fixed-size sample.


In [ ]:
cities = ['amsterdam', 'rotterdam', 'utrecht', 'den-haag']
CITY_SAMPLE_PAGES = 3
city_rows = []

for city in cities:
    total = search_total(city)
    sample = fetch_listings(city, pages=CITY_SAMPLE_PAGES, min_delay=2.0, max_delay=4.0, cooldown_base=30, long_pause_every=None)
    prices = pd.to_numeric(pd.Series([row.get('price') for row in sample]), errors='coerce')
    city_rows.append({
        'city': city.title(),
        'search_total': total,
        'sample_size': len(sample),
        'median_price': prices.median(),
        'mean_price': prices.mean(),
    })

df_cities = pd.DataFrame(city_rows).sort_values('median_price', ascending=False)
df_cities


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_cities.sort_values('search_total').plot(kind='barh', x='city', y='search_total', ax=axes[0], color='slategray', legend=False)
axes[0].set_title('Current Search Totals by City')
axes[0].set_xlabel('Listings')

df_cities.sort_values('median_price').plot(kind='barh', x='city', y='median_price', ax=axes[1], color='seagreen', legend=False)
axes[1].set_title(f'Median Asking Price by City ({CITY_SAMPLE_PAGES} pages sampled)')
axes[1].set_xlabel('Median Price (EUR)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()


## Export

In [ ]:
df.to_csv('amsterdam_listings.csv', index=False)
print('Saved to amsterdam_listings.csv')
